### Regresión lineal 
El Dataset implementado en cuestión es 'Tips', este contiene diferentes variables independientes que pueden ser utlizadas como varaibles predictoras, sin embargo, en este ejemplo intentaremos desarrollar un modelo lineal que solo se ajuste a una variable y que su aprendizaje sea en base a una correlación entre esta y su variable dependiente, varianzas iguales, distribuciones Gaussianas, etc.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
from scipy.stats import pearsonr, zscore
from sklearn.linear_model import LinearRegression


df = sns.load_dataset("tips")

# si bien definimos que vamos a emplear una sola variable predictora, graficaremos las etiquetas de es las varibales númericas

fig, ax = plt.subplots(2,2, figsize=(14,12))
sns.scatterplot(df,x="total_bill",y="tip",hue="sex",ax=ax[0,0])
sns.scatterplot(df,x="total_bill",y="tip",hue="smoker",ax=ax[0,1])
sns.scatterplot(df,x="total_bill",y="tip",hue="day",ax=ax[1,0])
sns.scatterplot(df,x="total_bill",y="tip",hue="time",ax=ax[1,1])
ax[0,0].grid("on")
ax[0,1].grid("on")
ax[1,0].grid("on")
ax[1,1].grid("on")
plt.show()

##### Conceptos estadísticos de un conjutno de datos
Varianza: medida de variabilidad que sebasa en dividir por el número de la muestra la sumatoria de los cuadrados entre las diferencias delos valores y la media.

Correlación(Pearson): valor estadístico que indica la fuerza ydirección de una asociación lineal entre dos variables númericas. 

Valores atípicos: valores que destacan por su notoria distinción(mayor o menor) a los demás del conjunto. Existen diferentes formas de hallar este tipo de valores, pero la más confiable es específica son los Valores Z: estos vendrían a serlos valores de la muestra representando cierta desviación éstandar. Con esta transformación,determinamos que los Valores Z mayores a 3 se consideran atípicos.La razón del uso de este valor como umbral proviene de la regla empírica, según la cual los datos que se encuentran dentro de 3 desviaciones éstandar con respecto a la media representan el 99.7% de los datos.Así podemos concluir con bastante seguridad que los datos que caen por afuera de este umbral son atípicos, ya que son distintos al 99.7% de los datos.


In [ ]:
corr, _ = pearsonr(df["total_bill"],df["tip"])

var_x = df["total_bill"].var()
var_y = df["tip"].var()

df_zscore = df[["total_bill","tip"]]

df_zscore["total_bill_zscore"] = zscore(df["total_bill"]).abs()
df_zscore["tip_zscore"] = zscore(df["tip"]).abs()

outliers = df_zscore.loc[(df_zscore["total_bill_zscore"] > 3) | (df_zscore["tip_zscore"] > 3),["total_bill","tip"]]

print("--------------------------------------------------------------------------------------------------")
print(f"Correlación pearsonr: {round(corr,2)} \n")
print(f"Varianza de la variable X: {round(var_x,2)} | Varianza de la variable Y: {round(var_x,2)} \n")
print(f"Valores atípicos: {outliers.shape[0]}")
print("--------------------------------------------------------------------------------------------------")

##### Medidas de distribución: nos indican si existe una distorsión en los datos y poder reconocer su comportamiento
Shapiro-Wilk: gráfico cuyo ejes 'X' y 'Y' son los valores Z y valores originales delconjunto. Es utilizado para visualizar algún posible sesgo en la dirección de los datos.

Kolmogorv-Smirnov: gráfico cuyo ejes 'X' y 'Y' son los valores del conjunto y la probalidad deencontrar un valor igual o menor a estos últimos, se interpreta en base a que los puntos de datosdeben seguir una figura en particular.


In [ ]:
df["total_bill_zscore"] = zscore(df["total_bill"])
df["tip_zscore"] = zscore(df["tip"])

sorted_total_bill = df["total_bill"].sort_values()
sorted_tip = df["tip"].sort_values()
percentage = np.arange(0,100,100 / df.shape[0])

fig = make_subplots(rows=2, cols=2)

fig.add_trace(go.Scatter(x=df["total_bill_zscore"],y=df["total_bill"],mode="markers",name="Cuentas totales"),row=1,col=1)
fig.add_trace(go.Scatter(x=df["tip_zscore"],y=df["tip"],mode="markers",name="Propinas"),row=1,col=2)
fig.add_trace(go.Scatter(x=sorted_total_bill.values,y=percentage,mode="markers",name="Cuentas totales"),row=2,col=1)
fig.add_trace(go.Scatter(x=sorted_tip.values,y=percentage,mode="markers",name="Propinas"),row=2,col=2)
fig.update_layout(height=600, width=850, title_text="Shapiro-wilk & Kolmogorov-smirnov")
fig

##### Luego de tener en cuenta los distintos resultados(correlación, varianzas, normalidad) se procede a generar el modelo lineal


In [19]:
linear_regression = LinearRegression()
 
total_bill = df["total_bill"].values.reshape((-1,1))

model = linear_regression.fit(total_bill,df["tip"])

# ------- regresión de nuevos datos -------

objects = np.array([[28.15],[12.5],[3.8],[8.25],[19.5],[32.7],[40.9],[45]])

predicts = model.predict(objects)

##### Dashboard que refleja el análisis estadístico para la creación del modelo y como este se ajusta a los datos

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div(id="body",className="e3_body",children=[
    html.H1("Propinas",id="titulo",className="e3_title"),
    html.Div(id="dashboard",className="e3_dashboard",children=[
        html.Div(id="column-1",className="e3_column_1",children=[
            dcc.Dropdown(id="dropdown",className="e3_dropdown",
                        options=[
                            {"label":"Cuentas totales","value":"total_bill"},
                            {"label":"Propinas","value":"tip"}
                        ],
                        value="total_bill",
                        multi=False,
                        clearable=False),
            html.Div(className="e3_div_graphs",children=[
                dcc.Graph(id="graph-1",className="e3_graphs",figure={}), 
                dcc.Graph(id="graph-2",className="e3_graphs",figure={})
            ])
        ]),
        html.Div(id="column-2",className="e3_column_2",children=[
            html.Div(id="stats_div",className="e3_stats_div",children=[
                html.Div(id="var_x",className="e3_stats",children=[html.P(f"Varianza X: {round(var_x,2)}",style={"font-size":"1em"})]),
                html.Div(id="var_y",className="e3_stats",children=[html.P(f"Varianza Y: {round(var_y,2)}",style={"font-size":"0.98em"})])
            ]),
            html.Div(f"Correlación: {round(corr,2)}",className="e3_corr",id="corr"),
            dcc.Graph(id="graph-3",className="e3_graph_3",figure={})
        ]),
        html.P("Conclusión: si bien, especificando este caso, suena lógico que mientras mayor sea el pago total(variblae X) mayor sea la propina(variable Y), pero esto puede llegar a traer hipótesis erróneas. Si bien el valor de correlación que obutvimos es óptimo(0.68), vimos casos donde algunas variables difieren bastante del patrón, esto debido a que anteriormente mencionamos que el DataSet contiene mas variables independientes que seguramente influyan en ese sesgo en particular, por ejemplo: los fines de semanas o en el horario de cena la propina es mayor. Por ello, es importante realizar constatemente análisis en cada elemento del conjunto, ya que el origen del sesgo sigue un patrón y es predecible.",className="e3_p")
    ])
])

@app.callback(
    [Output(component_id="graph-1",component_property="figure"),
    Output(component_id="graph-2",component_property="figure"),
    Output(component_id="graph-3",component_property="figure")],
    [Input(component_id="dropdown",component_property="value")]
)

def update_dash(slct_var):
    
    mean = df[slct_var].mean()
    median = df[slct_var].median()
    
    extr_list = [0]
    
    var_title = "Cuentas totales"
    
    if slct_var == "tip":
        extr_list.append(60)
        var_title = "Propinas"
    elif slct_var == "total_bill":
        extr_list.append(40)
        var_title = "Cuentas totales"
    
    histplot = go.Figure(go.Histogram(x=df[slct_var],name="Distribución"))
    histplot.add_trace(go.Scatter(x=[mean,mean],y=extr_list,mode="lines+markers",marker_color="red",name="Media"))
    histplot.add_trace(go.Scatter(x=[median,median],y=extr_list,mode="lines+markers",marker_color="green",name="Mediana"))
    histplot.update_layout(title="Histograma",xaxis_title=var_title)
        
    df["zscore"] = zscore(df[slct_var])
    shapiro_wilk = px.scatter(df,x="zscore",y=slct_var)
    shapiro_wilk.update_layout(title="Shapiro-wilk",xaxis_title="Valores Z",yaxis_title=var_title)
    
    scatter = go.Figure()
    scatter.add_trace(go.Scatter(x=df["total_bill"],y=df["tip"],mode="markers",marker_color="blue",name="Propinas reales"))
    scatter.add_trace(go.Scatter(x=objects.reshape(-1),y=predicts,mode="lines+markers",marker_color="red",name="Predicciones"))
    scatter.update_layout(title="Regresión Lineal",xaxis_title="Cuentas totales",yaxis_title="Propinas")

    return histplot, shapiro_wilk, scatter

if __name__ == "__main__":
    app.run_server(debug=False)

##### Error de bías: también denominado sesgo, es la diferencia entre la predicciones esperados delmodelo y los valores verdaderos. Sucede principalmente en algoritmos paramétricos que requierenciertas representaciones o direcciones específicas sin distorsión en los datos, de forma que si hayun sesgo que afecte la validez y precisión de los resultados se descubra con antelación y se loreduzca.
Alto bías: requiere más suposiciones a la hora de estimar la función objetivo, ejemplosde algoritmos: Regresión Lineal, Regresión Logística, algoritmos de series temporales.
